# Rigorous Statistical Evaluation of ATMS Hallucination Meter — Iter 1 Results

This notebook evaluates the **Bounded ATMS Hallucination Meter** against CoT, LINC, and ProbLog baselines on 48 reasoning instances (10 CLUTRR, 8 bAbI, 30 custom).

Key metrics computed:
- **Empty-environment precision** (Wilson CI): accuracy when assumption_load = 0
- **Risk-coverage AUC** with bootstrap confidence intervals
- **Coverage-Weighted Accuracy (CWA)** at multiple coverage levels
- **Spearman correlation** between assumption_load and error rate
- **AUROC** of assumption_load as an error predictor
- **L1 degradation theorem** bounds
- **ATMS vs ProbLog divergence** analysis

The central finding: ATMS achieves high precision on empty-environment instances (load=0) but abstains on 77% of instances where assumption_load=inf, yielding a lower overall accuracy but better selective risk.

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# statsmodels is NOT pre-installed on Colab — always install
_pip('statsmodels==0.14.6')

# Core scientific packages — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'scikit-learn==1.6.1')


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import gc
import json
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from sklearn.metrics import roc_auc_score

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-70ae4e-the-empty-environment-test-calibration-f/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
raw_data = load_data()
print(f"Loaded datasets: {[d['dataset'] for d in raw_data['datasets']]}")
print(f"Examples per dataset: {[len(d['examples']) for d in raw_data['datasets']]}")

Loaded datasets: ['clutrr', 'babi', 'custom']
Examples per dataset: [4, 3, 3]


## Config

Tunable parameters for the evaluation. Set to minimum values for a quick demo run.

In [5]:
# Bootstrap samples — increase for more stable CIs (original: 1000)
N_BOOT = 200

# Coverage targets for CWA analysis
COVERAGE_TARGETS = [0.10, 0.229, 0.50, 1.00]

# Figures output directory
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

## Helper Functions

Wilson confidence interval, bootstrap risk-coverage AUC, and bootstrap AUROC.

In [6]:
def wilson_ci(count: int, nobs: int, alpha: float = 0.05) -> tuple:
    if nobs == 0:
        return (0.0, 0.0)
    lo, hi = proportion_confint(count, nobs, alpha=alpha, method="wilson")
    return float(lo), float(hi)


def bootstrap_arc(errors, scores, n_boot=N_BOOT, seed=42):
    """Return (auc, ci_lower, ci_upper) for the risk-coverage curve via bootstrap."""
    n = len(errors)
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        e_b = errors[idx]
        s_b = scores[idx]
        order = np.argsort(s_b)
        e_sorted = e_b[order]
        risks = np.cumsum(e_sorted) / np.arange(1, n + 1)
        aucs.append(float(np.mean(risks)))
    order_full = np.argsort(scores)
    e_sorted_full = errors[order_full]
    risks_full = np.cumsum(e_sorted_full) / np.arange(1, n + 1)
    auc = float(np.mean(risks_full))
    ci_lo = float(np.percentile(aucs, 2.5))
    ci_hi = float(np.percentile(aucs, 97.5))
    return auc, ci_lo, ci_hi


def auroc_bootstrap(y_true, scores, n_boot=N_BOOT, seed=42):
    """AUROC with bootstrap CI."""
    n = len(y_true)
    rng = np.random.default_rng(seed)

    if len(np.unique(y_true)) < 2:
        print("AUROC degenerate (single class) — returning 0.5")
        return 0.5, 0.5, 0.5

    auc = float(roc_auc_score(y_true, scores))
    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        y_b = y_true[idx]
        s_b = scores[idx]
        if len(np.unique(y_b)) < 2:
            continue
        boot_aucs.append(float(roc_auc_score(y_b, s_b)))
    if not boot_aucs:
        return auc, auc, auc
    return auc, float(np.percentile(boot_aucs, 2.5)), float(np.percentile(boot_aucs, 97.5))

## Data Loading

Parse the raw dataset into per-instance records with typed fields (assumption_load as float/inf, boolean flags, correct indicators).

In [7]:
def parse_records(raw):
    records = []
    for ds_block in raw["datasets"]:
        ds_name = ds_block["dataset"]
        for ex in ds_block["examples"]:
            r = dict(ex)
            r["dataset"] = ds_name
            # parse assumption_load
            al = r.get("metadata_assumption_load", "0")
            try:
                r["assumption_load"] = np.inf if str(al).lower() == "inf" else float(al)
            except (ValueError, TypeError):
                print(f"Unparseable assumption_load={al!r}, skipping example")
                continue
            # boolean strings
            r["empty_env_derivable"] = str(r.get("metadata_empty_env_derivable", "False")) == "True"
            r["l1_grounding_verified"] = str(r.get("metadata_l1_grounding_verified", "False")) == "True"
            # correct flags
            gold = r.get("output", "")
            r["correct_our_method"] = r.get("predict_our_method", "") == gold
            r["correct_cot"] = r.get("predict_cot", "") == gold
            r["correct_linc"] = r.get("predict_linc", "") == gold
            r["correct_problog"] = r.get("predict_problog", "") == gold
            # abstain: assumption_load == inf
            r["abstained"] = np.isinf(r["assumption_load"])
            records.append(r)
    print(f"Loaded {len(records)} instances from {len(raw['datasets'])} datasets")
    return records

records = parse_records(raw_data)
n = len(records)

Loaded 10 instances from 3 datasets


## Section 2: Proportions with Wilson Confidence Intervals

Compute accuracy/precision/abstain-rate for ATMS and baselines, each with a 95% Wilson CI. Wilson CIs are preferred over Wald for small samples because they stay within [0,1].

In [8]:
def compute_proportions(records):
    n = len(records)
    rows = []

    def add(metric, condition_fn, correct_fn):
        subset = [r for r in records if condition_fn(r)]
        count = sum(1 for r in subset if correct_fn(r))
        nobs = len(subset)
        p = count / nobs if nobs else 0.0
        lo, hi = wilson_ci(count, nobs)
        rows.append({"metric": metric, "count": count, "nobs": nobs,
                     "proportion": round(p, 6), "ci_lower": round(lo, 6), "ci_upper": round(hi, 6)})

    # empty-env precision for our method
    add("empty_env_precision_our_method",
        lambda r: r["empty_env_derivable"],
        lambda r: r["correct_our_method"])

    # overall accuracy (answered only)
    add("accuracy_answered_our_method",
        lambda r: not r["abstained"],
        lambda r: r["correct_our_method"])

    # overall accuracy all instances
    add("accuracy_all_our_method",
        lambda r: True,
        lambda r: r["correct_our_method"])

    # abstain rate
    abstain_count = sum(1 for r in records if r["abstained"])
    p_abs = abstain_count / n
    lo, hi = wilson_ci(abstain_count, n)
    rows.append({"metric": "abstain_rate", "count": abstain_count, "nobs": n,
                 "proportion": round(p_abs, 6), "ci_lower": round(lo, 6), "ci_upper": round(hi, 6)})

    # l1 grounding rate
    l1_count = sum(1 for r in records if r["l1_grounding_verified"])
    lo, hi = wilson_ci(l1_count, n)
    rows.append({"metric": "l1_grounding_rate", "count": l1_count, "nobs": n,
                 "proportion": round(l1_count / n, 6), "ci_lower": round(lo, 6), "ci_upper": round(hi, 6)})

    # baselines (all instances, no abstentions)
    for m in ["cot", "linc", "problog"]:
        add(f"accuracy_all_{m}",
            lambda r: True,
            lambda r, m=m: r[f"correct_{m}"])

    return rows

proportions_ci = compute_proportions(records)
for row in proportions_ci:
    print(f"{row['metric']:40s}  {row['proportion']:.3f}  [{row['ci_lower']:.3f}, {row['ci_upper']:.3f}]  (n={row['nobs']})")

empty_env_precision_our_method            1.000  [0.566, 1.000]  (n=5)
accuracy_answered_our_method              1.000  [0.566, 1.000]  (n=5)
accuracy_all_our_method                   0.500  [0.237, 0.763]  (n=10)
abstain_rate                              0.500  [0.237, 0.763]  (n=10)
l1_grounding_rate                         1.000  [0.722, 1.000]  (n=10)
accuracy_all_cot                          0.800  [0.490, 0.943]  (n=10)
accuracy_all_linc                         0.800  [0.490, 0.943]  (n=10)
accuracy_all_problog                      0.900  [0.596, 0.982]  (n=10)


## Section 3: Risk-Coverage Curves and AUC

ATMS uses assumption_load as a confidence score: instances with load=0 are answered first (lowest coverage), those with load=inf last. The risk-coverage curve shows selective error rate as coverage threshold increases. Baselines lack per-instance confidence, so their curves are flat lines at their error rate.

In [9]:
def compute_risk_coverage(records):
    n = len(records)
    # our_method: score = assumption_load (inf → 1e9)
    load_scores = np.array([min(r["assumption_load"], 1e9) for r in records])
    errors_ours = np.array([0.0 if r["correct_our_method"] else 1.0 for r in records])

    auc_our, ci_lo_our, ci_hi_our = bootstrap_arc(errors_ours, load_scores)

    # baselines: constant score → flat line at error rate
    arc_results = {"our_method": {"auc": round(auc_our, 6),
                                   "ci_lower": round(ci_lo_our, 6),
                                   "ci_upper": round(ci_hi_our, 6)}}
    flat_arcs = {}
    for m in ["cot", "linc", "problog"]:
        errors_b = np.array([0.0 if r[f"correct_{m}"] else 1.0 for r in records])
        flat_err = float(errors_b.mean())
        flat_arcs[m] = flat_err
        arc_results[m] = {"auc": round(flat_err, 6),
                          "ci_lower": round(flat_err, 6),
                          "ci_upper": round(flat_err, 6),
                          "note": "Baselines lack per-instance confidence; ARC = flat error rate"}

    # curve data for plotting
    order = np.argsort(load_scores)
    e_sorted = errors_ours[order]
    coverages = np.arange(1, n + 1) / n
    risks = np.cumsum(e_sorted) / np.arange(1, n + 1)

    return arc_results, {"coverages": coverages, "risks": risks, "flat_arcs": flat_arcs,
                         "errors_ours": errors_ours, "load_scores": load_scores}

arc_results, arc_data = compute_risk_coverage(records)
print("Risk-Coverage AUC:")
for m, v in arc_results.items():
    print(f"  {m}: {v['auc']:.4f} [{v['ci_lower']:.4f}, {v['ci_upper']:.4f}]")

Risk-Coverage AUC:
  our_method: 0.1772 [0.0311, 0.5190]
  cot: 0.2000 [0.2000, 0.2000]
  linc: 0.2000 [0.2000, 0.2000]
  problog: 0.1000 [0.1000, 0.1000]


In [10]:
def plot_risk_coverage(arc_data, n):
    fig, ax = plt.subplots(figsize=(7, 5))
    coverages = arc_data["coverages"]
    risks = arc_data["risks"]
    flat_arcs = arc_data["flat_arcs"]
    errors_ours = arc_data["errors_ours"]
    load_scores = arc_data["load_scores"]

    # bootstrap band
    rng = np.random.default_rng(42)
    boot_risks = []
    for _ in range(N_BOOT):
        idx = rng.choice(n, size=n, replace=True)
        e_b = errors_ours[idx]
        s_b = load_scores[idx]
        order_b = np.argsort(s_b)
        boot_risks.append(np.cumsum(e_b[order_b]) / np.arange(1, n + 1))
    boot_risks = np.array(boot_risks)
    lo_band = np.percentile(boot_risks, 2.5, axis=0)
    hi_band = np.percentile(boot_risks, 97.5, axis=0)

    ax.plot(coverages, risks, color="steelblue", linewidth=2, label="ATMS (ours)")
    ax.fill_between(coverages, lo_band, hi_band, alpha=0.25, color="steelblue")

    colors = {"cot": "darkorange", "linc": "green", "problog": "red"}
    for m, err in flat_arcs.items():
        ax.axhline(err, linestyle="--", color=colors[m], linewidth=1.5, label=f"{m.upper()} (flat={err:.3f})")

    ax.set_xlabel("Coverage", fontsize=12)
    ax.set_ylabel("Selective Risk (error rate)", fontsize=12)
    ax.set_title(f"Risk-Coverage Curves (n={n})", fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    path = str(FIGURES_DIR / "risk_coverage.png")
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"Saved {path}")
    return path

fig_rc = plot_risk_coverage(arc_data, n)

Saved figures/risk_coverage.png


/tmp/claude-1000/claude-1000/ipykernel_3425800/87690469.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 4: Coverage-Weighted Accuracy (CWA)

CWA measures accuracy at different coverage levels: select only the k most-confident instances (lowest assumption_load) and compute accuracy. This shows whether ATMS selects correctly when it does answer.

In [11]:
def compute_cwa(records, coverage_targets):
    n = len(records)
    load_scores = [min(r["assumption_load"], 1e9) for r in records]
    order = sorted(range(n), key=lambda i: load_scores[i])

    rows = []
    for c in coverage_targets:
        k = max(1, math.floor(c * n))
        subset_idx = order[:k]
        subset = [records[i] for i in subset_idx]
        for method in ["our_method", "cot", "linc", "problog"]:
            if method == "our_method":
                correct = sum(1 for r in subset if r["correct_our_method"])
            else:
                # baselines answer all → CWA at any coverage = sample accuracy
                # for consistent comparison, use the same k instances
                correct = sum(1 for r in subset if r[f"correct_{method}"])
            acc = correct / k
            lo, hi = wilson_ci(correct, k)
            rows.append({"method": method, "coverage_target": round(c, 4), "n_answered": k,
                         "accuracy": round(acc, 6), "ci_lower": round(lo, 6), "ci_upper": round(hi, 6)})
    return rows

cwa_table = compute_cwa(records, COVERAGE_TARGETS)
print(f"{'method':12s} {'coverage':9s} {'n':4s} {'accuracy':9s}")
print("-" * 40)
for row in cwa_table:
    print(f"{row['method']:12s} {row['coverage_target']:9.3f} {row['n_answered']:4d} {row['accuracy']:9.3f}")

method       coverage  n    accuracy 
----------------------------------------
our_method       0.100    1     1.000
cot              0.100    1     1.000
linc             0.100    1     1.000
problog          0.100    1     1.000
our_method       0.229    2     1.000
cot              0.229    2     1.000
linc             0.229    2     0.500
problog          0.229    2     0.500
our_method       0.500    5     1.000
cot              0.500    5     0.800
linc             0.500    5     0.600
problog          0.500    5     0.800
our_method       1.000   10     0.500
cot              1.000   10     0.800
linc             1.000   10     0.800
problog          1.000   10     0.900


In [12]:
def plot_cwa(cwa_table):
    fig, ax = plt.subplots(figsize=(7, 5))
    methods = ["our_method", "cot", "linc", "problog"]
    colors = {"our_method": "steelblue", "cot": "darkorange", "linc": "green", "problog": "red"}
    labels = {"our_method": "ATMS (ours)", "cot": "CoT", "linc": "LINC", "problog": "ProbLog"}

    for m in methods:
        rows = [r for r in cwa_table if r["method"] == m]
        xs = [r["coverage_target"] for r in rows]
        ys = [r["accuracy"] for r in rows]
        ax.plot(xs, ys, marker="o", color=colors[m], linewidth=1.8, label=labels[m])

    ax.set_xlabel("Coverage Target", fontsize=12)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title("Coverage-Weighted Accuracy vs Coverage Level", fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    path = str(FIGURES_DIR / "cwa_vs_coverage.png")
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"Saved {path}")
    return path

fig_cwa = plot_cwa(cwa_table)

Saved figures/cwa_vs_coverage.png


/tmp/claude-1000/claude-1000/ipykernel_3425800/4282221799.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 5: Assumption-Load vs Error Analysis

Two tests of assumption_load as an error predictor:
1. **Spearman ρ**: monotonic correlation between load and error indicator
2. **AUROC**: area under the ROC curve treating load as a binary error classifier

In [13]:
def compute_load_error_analysis(records):
    sentinel = 10.0
    load_clipped = np.array([sentinel if np.isinf(r["assumption_load"]) else r["assumption_load"]
                              for r in records])
    errors = np.array([0.0 if r["correct_our_method"] else 1.0 for r in records])

    rho, pval = stats.spearmanr(load_clipped, errors)

    auroc, ci_lo, ci_hi = auroc_bootstrap(errors, load_clipped)

    return {
        "spearman_rho": round(float(rho), 6),
        "spearman_p": round(float(pval), 6),
        "auroc_assumption_load": {
            "auroc": round(auroc, 6),
            "ci_lower": round(ci_lo, 6),
            "ci_upper": round(ci_hi, 6),
        },
    }


def compute_calibration(records):
    strata = {
        "load=0 (empty-env)": lambda r: r["assumption_load"] == 0.0,
        "load=1": lambda r: r["assumption_load"] == 1.0,
        "load=inf": lambda r: np.isinf(r["assumption_load"]),
    }
    rows = []
    for label, fn in strata.items():
        subset = [r for r in records if fn(r)]
        n = len(subset)
        n_correct = sum(1 for r in subset if r["correct_our_method"])
        frac = n_correct / n if n else 0.0
        lo, hi = wilson_ci(n_correct, n)
        underpowered = n < 5
        rows.append({"stratum": label, "n": n, "n_correct": n_correct,
                     "fraction_correct": round(frac, 6),
                     "ci_lower": round(lo, 6), "ci_upper": round(hi, 6),
                     "underpowered": underpowered})
        if underpowered:
            print(f"WARNING: Stratum '{label}' underpowered (n={n} < 5)")
    return rows

load_analysis = compute_load_error_analysis(records)
cal_strata = compute_calibration(records)
print(f"Spearman rho={load_analysis['spearman_rho']:.4f}  p={load_analysis['spearman_p']:.4f}")
print(f"AUROC(load)={load_analysis['auroc_assumption_load']['auroc']:.4f} "
      f"[{load_analysis['auroc_assumption_load']['ci_lower']:.4f}, "
      f"{load_analysis['auroc_assumption_load']['ci_upper']:.4f}]")

Spearman rho=1.0000  p=0.0000
AUROC(load)=1.0000 [1.0000, 1.0000]


In [14]:
def plot_calibration(cal_strata):
    fig, ax = plt.subplots(figsize=(7, 5))
    labels = [r["stratum"] for r in cal_strata]
    fracs = [r["fraction_correct"] for r in cal_strata]
    los = [r["fraction_correct"] - r["ci_lower"] for r in cal_strata]
    his = [r["ci_upper"] - r["fraction_correct"] for r in cal_strata]
    ns = [r["n"] for r in cal_strata]

    x = np.arange(len(labels))
    bars = ax.bar(x, fracs, color=["steelblue", "darkorange", "tomato"], alpha=0.8,
                  yerr=[los, his], capsize=5, error_kw={"linewidth": 1.5})

    for bar, n_val in zip(bars, ns):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
                f"n={n_val}", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel("Fraction Correct", fontsize=12)
    ax.set_title("Accuracy by Assumption-Load Stratum", fontsize=13)
    ax.set_ylim(0, 1.15)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    path = str(FIGURES_DIR / "calibration_by_load.png")
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"Saved {path}")
    return path

fig_cal = plot_calibration(cal_strata)

Saved figures/calibration_by_load.png


/tmp/claude-1000/claude-1000/ipykernel_3425800/2355221026.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 6: L1 Degradation Theorem

Derives a formal lower bound on empty-environment precision as a function of per-atom grounding error rate (ε_L1) and derivation chain length (k). With all instances L1-verified (ε_L1=0), the bound collapses to 1.0; example instantiation shows ε_L1=0.04, k=3 → bound=0.885.

In [15]:
def l1_degradation_theorem(records):
    observed_eps = 0.0 if all(r["l1_grounding_verified"] for r in records) else (
        sum(1 for r in records if not r["l1_grounding_verified"]) / len(records)
    )
    return {
        "statement": (
            "Let epsilon_L1 be the per-atom L1 grounding error rate (probability any given atom "
            "is incorrectly grounded), with grounding errors independent across atoms. "
            "For an empty-environment derivation requiring a chain of k atoms: "
            "P(derivation fully correct) >= (1 - epsilon_L1)^k. "
            "Therefore empty-environment precision >= (1 - epsilon_L1)^k_max, "
            "where k_max is the maximum chain length in the derivation."
        ),
        "union_bound": "precision_EE >= 1 - k * epsilon_L1",
        "jensen_bound": "E[precision_EE] >= exp(-epsilon_L1 * k_bar) for mean depth k_bar",
        "instantiation_04_k3": round((1 - 0.04) ** 3, 6),
        "instantiation_04_k3_union_bound": round(1 - 3 * 0.04, 6),
        "observed_epsilon_l1": round(observed_eps, 6),
        "note": (
            "With observed epsilon_L1=0.0 (all instances L1-verified), the bound collapses to 1.0. "
            "The bound is non-trivial for epsilon_L1 > 0: e.g., epsilon_L1=0.04 (Path-of-Thoughts "
            "95.9% triplet accuracy), k=3 hops → lower bound = (0.96)^3 = 0.885."
        ),
    }

l1_theorem = l1_degradation_theorem(records)
print(f"observed_epsilon_l1 = {l1_theorem['observed_epsilon_l1']}")
print(f"bound(eps=0.04, k=3) = {l1_theorem['instantiation_04_k3']}")
print(f"union bound          = {l1_theorem['instantiation_04_k3_union_bound']}")

observed_epsilon_l1 = 0.0
bound(eps=0.04, k=3) = 0.884736
union bound          = 0.88


## Section 7: ATMS vs ProbLog Divergence

Identifies instances where ATMS and ProbLog predictions differ, and measures which method was correct on those divergent cases.

In [16]:
def compute_divergence(records):
    divergent = [r for r in records if r.get("predict_our_method") != r.get("predict_problog")]
    print(f"Found {len(divergent)} ATMS-vs-ProbLog divergent instances")
    rows = []
    for r in divergent:
        gold = r.get("output", "")
        atms_pred = r.get("predict_our_method", "")
        prob_pred = r.get("predict_problog", "")
        rows.append({
            "dataset": r["dataset"],
            "input_snippet": r.get("input", "")[:100],
            "gold": gold,
            "predict_atms": atms_pred,
            "predict_problog": prob_pred,
            "atms_correct": atms_pred == gold,
            "problog_correct": prob_pred == gold,
        })
    atms_correct_count = sum(1 for r in rows if r["atms_correct"])
    problog_correct_count = sum(1 for r in rows if r["problog_correct"])
    return rows, atms_correct_count, problog_correct_count

divergence_rows, atms_div_correct, problog_div_correct = compute_divergence(records)
print(f"ATMS correct on divergent: {atms_div_correct}/{len(divergence_rows)}")
print(f"ProbLog correct on divergent: {problog_div_correct}/{len(divergence_rows)}")

Found 6 ATMS-vs-ProbLog divergent instances
ATMS correct on divergent: 1/6
ProbLog correct on divergent: 5/6


## Results Summary

Key metrics printed in a readable table.

In [17]:
print("=" * 65)
print("ATMS HALLUCINATION METER — EVALUATION SUMMARY")
print("=" * 65)
print(f"n_instances:              {n}")
print()

# Proportions
ee = proportions_ci[0]
ans = proportions_ci[1]
all_atms = proportions_ci[2]
abst = proportions_ci[3]
cot_acc = proportions_ci[5]
linc_acc = proportions_ci[6]
prob_acc = proportions_ci[7]

print(f"{'Metric':<38} {'Value':>7}  {'95% Wilson CI'}")
print("-" * 65)
print(f"{'Empty-env precision (ATMS)':<38} {ee['proportion']:>7.3f}  [{ee['ci_lower']:.3f}, {ee['ci_upper']:.3f}]  n={ee['nobs']}")
print(f"{'Accuracy (answered only, ATMS)':<38} {ans['proportion']:>7.3f}  [{ans['ci_lower']:.3f}, {ans['ci_upper']:.3f}]  n={ans['nobs']}")
print(f"{'Accuracy (all, ATMS)':<38} {all_atms['proportion']:>7.3f}  [{all_atms['ci_lower']:.3f}, {all_atms['ci_upper']:.3f}]  n={all_atms['nobs']}")
print(f"{'Abstain rate (ATMS)':<38} {abst['proportion']:>7.3f}")
print(f"{'Accuracy (all, CoT)':<38} {cot_acc['proportion']:>7.3f}")
print(f"{'Accuracy (all, LINC)':<38} {linc_acc['proportion']:>7.3f}")
print(f"{'Accuracy (all, ProbLog)':<38} {prob_acc['proportion']:>7.3f}")
print()

# AUC
arc = arc_results
print(f"{'Risk-Coverage AUC':<38} {'Value':>7}  {'95% Bootstrap CI'}")
print("-" * 65)
print(f"{'ATMS':<38} {arc['our_method']['auc']:>7.4f}  [{arc['our_method']['ci_lower']:.4f}, {arc['our_method']['ci_upper']:.4f}]")
print(f"{'CoT (flat)':<38} {arc['cot']['auc']:>7.4f}")
print(f"{'LINC (flat)':<38} {arc['linc']['auc']:>7.4f}")
print(f"{'ProbLog (flat)':<38} {arc['problog']['auc']:>7.4f}")
print()

# Load-error
print(f"Spearman rho (load vs error): {load_analysis['spearman_rho']:.4f}  p={load_analysis['spearman_p']:.4f}")
au = load_analysis['auroc_assumption_load']
print(f"AUROC(load):  {au['auroc']:.4f}  [{au['ci_lower']:.4f}, {au['ci_upper']:.4f}]")
print()

# Calibration
print(f"{'Calibration by stratum':<30} {'n':>5} {'correct':>8} {'frac':>7}")
print("-" * 55)
for s in cal_strata:
    flag = " (underpowered)" if s["underpowered"] else ""
    print(f"  {s['stratum']:<28} {s['n']:>5} {s['n_correct']:>8} {s['fraction_correct']:>7.3f}{flag}")
print()

# L1 theorem
print(f"L1 observed epsilon: {l1_theorem['observed_epsilon_l1']}")
print(f"L1 bound (eps=0.04, k=3): {l1_theorem['instantiation_04_k3']}")
print()

# Divergence
print(f"ATMS vs ProbLog divergent: {len(divergence_rows)}")
print(f"  ATMS correct on divergent:    {atms_div_correct}")
print(f"  ProbLog correct on divergent: {problog_div_correct}")
print("=" * 65)

# Summary bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: accuracy comparison
methods = ["ATMS\n(answered)", "ATMS\n(all)", "CoT", "LINC", "ProbLog"]
accs = [ans['proportion'], all_atms['proportion'], cot_acc['proportion'],
        linc_acc['proportion'], prob_acc['proportion']]
colors_bar = ["steelblue", "lightsteelblue", "darkorange", "green", "red"]
axes[0].bar(methods, accs, color=colors_bar, alpha=0.85)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy Comparison")
axes[0].set_ylim(0, 1.1)
axes[0].grid(axis="y", alpha=0.3)
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

# Right: risk-coverage AUC
methods2 = ["ATMS", "CoT", "LINC", "ProbLog"]
aucs = [arc['our_method']['auc'], arc['cot']['auc'], arc['linc']['auc'], arc['problog']['auc']]
colors2 = ["steelblue", "darkorange", "green", "red"]
axes[1].bar(methods2, aucs, color=colors2, alpha=0.85)
axes[1].set_ylabel("Risk-Coverage AUC (lower=better)")
axes[1].set_title("Risk-Coverage AUC")
axes[1].set_ylim(0, 0.6)
axes[1].grid(axis="y", alpha=0.3)
for i, v in enumerate(aucs):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "summary.png"), dpi=150)
plt.show()
print("Summary figure saved.")

ATMS HALLUCINATION METER — EVALUATION SUMMARY
n_instances:              10

Metric                                   Value  95% Wilson CI
-----------------------------------------------------------------
Empty-env precision (ATMS)               1.000  [0.566, 1.000]  n=5
Accuracy (answered only, ATMS)           1.000  [0.566, 1.000]  n=5
Accuracy (all, ATMS)                     0.500  [0.237, 0.763]  n=10
Abstain rate (ATMS)                      0.500
Accuracy (all, CoT)                      0.800
Accuracy (all, LINC)                     0.800
Accuracy (all, ProbLog)                  0.900

Risk-Coverage AUC                        Value  95% Bootstrap CI
-----------------------------------------------------------------
ATMS                                    0.1772  [0.0311, 0.5190]
CoT (flat)                              0.2000
LINC (flat)                             0.2000
ProbLog (flat)                          0.1000

Spearman rho (load vs error): 1.0000  p=0.0000
AUROC(load):  1.0

Summary figure saved.


/tmp/claude-1000/claude-1000/ipykernel_3425800/1257025571.py:92: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
